In [1]:
import os
from google import genai
from qdrant_client import QdrantClient

# 1. เช็ค Qdrant Client
def check_qdrant_methods():
    print("--- Checking QdrantClient Methods ---")
    # สมมติ URL ตามที่คุณใช้ใน ingest.py
    url = "http://localhost:6333" 
    client = QdrantClient(url=url)
    
    # ดึงรายชื่อเมธอดทั้งหมดออกมา
    methods = [m for m in dir(client) if callable(getattr(client, m)) and not m.startswith("_")]
    
    # เช็คว่ามี .search หรือไม่
    if "search" in methods:
        print("✅ พบเมธอด .search() ใน QdrantClient")
    else:
        print("❌ ไม่พบเมธอด .search()! ลองเช็คชื่อเมธอดด้านล่าง:")
        
    print(f"รายการเมธอดทั้งหมด: {', '.join(methods)}")

# 2. เช็ค Gemini Client (เผื่อไว้)
def check_gemini_methods():
    print("\n--- Checking Gemini Client Methods ---")
    api_key = os.environ.get("GEMINI_API_KEY")
    if api_key:
        client = genai.Client(api_key=api_key)
        methods = [m for m in dir(client) if callable(getattr(client, m)) and not m.startswith("_")]
        print(f"รายการเมธอด Gemini Client: {', '.join(methods)}")
    else:
        print("ข้ามการเช็ค Gemini (ไม่ได้ตั้งค่า API Key)")

if __name__ == "__main__":
    check_qdrant_methods()
    check_gemini_methods()

--- Checking QdrantClient Methods ---
❌ ไม่พบเมธอด .search()! ลองเช็คชื่อเมธอดด้านล่าง:
รายการเมธอดทั้งหมด: add, batch_update_points, clear_payload, close, cluster_collection_update, cluster_status, cluster_telemetry, collection_cluster_info, collection_exists, count, create_collection, create_full_snapshot, create_payload_index, create_shard_key, create_shard_snapshot, create_snapshot, create_vector_name, delete, delete_collection, delete_full_snapshot, delete_payload, delete_payload_index, delete_shard_key, delete_shard_snapshot, delete_snapshot, delete_vector_name, delete_vectors, facet, get_aliases, get_collection, get_collection_aliases, get_collections, get_embedding_size, get_fastembed_sparse_vector_params, get_fastembed_vector_params, get_optimizations, get_sparse_vector_field_name, get_vector_field_name, info, list_full_snapshots, list_image_models, list_late_interaction_multimodal_models, list_late_interaction_text_models, list_shard_keys, list_shard_snapshots, list_snapshots

C:\Users\BM MONEY\AppData\Local\Temp\ipykernel_26760\2692702064.py:29: UserWarning: Agents usage is experimental and may change in future versions.
  methods = [m for m in dir(client) if callable(getattr(client, m)) and not m.startswith("_")]
C:\Users\BM MONEY\AppData\Local\Temp\ipykernel_26760\2692702064.py:29: UserWarning: Interactions usage is experimental and may change in future versions.
  methods = [m for m in dir(client) if callable(getattr(client, m)) and not m.startswith("_")]


In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

# Local Qdrant (ไม่ต้องเปิด Docker)
client = QdrantClient(":memory:")

# Create Collection
client.create_collection(
    collection_name="test_docs",
    vectors_config=VectorParams(
        size=4,
        distance=Distance.COSINE
    )
)

# Insert vectors
client.upsert(
    collection_name="test_docs",
    points=[
        PointStruct(
            id=1,
            vector=[0.9, 0.1, 0.1, 0.1],
            payload={"text": "Diabetes Guideline"}
        ),
        PointStruct(
            id=2,
            vector=[0.8, 0.2, 0.1, 0.1],
            payload={"text": "Hypertension Guideline"}
        ),
        PointStruct(
            id=3,
            vector=[0.1, 0.9, 0.1, 0.1],
            payload={"text": "Pneumonia Guideline"}
        )
    ]
)

print("Insert Success")

Insert Success


In [4]:
result = client.query_points(
    collection_name="test_docs",
    query=[0.85, 0.15, 0.1, 0.1],
    limit=2
)

for point in result.points:
    print(
        point.id,
        point.score,
        point.payload
    )

1 0.9979749155678157 {'text': 'Diabetes Guideline'}
2 0.9975694184303856 {'text': 'Hypertension Guideline'}


In [7]:
from qdrant_client.http import models

print(hasattr(models, "QueryVector"))

False


In [8]:
import numpy as np
from google import genai
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import os
from dotenv import load_dotenv

load_dotenv()

# --- Configuration ---
API_KEY = os.getenv("GEMINI_API_KEY") 
client = genai.Client(api_key=API_KEY)

# Context สำหรับทดสอบข้ามภาษา (Medical Context)
th_text = "การวินิจฉัยโรคติดเชื้อเฉียบพลันระบบหายใจในเด็ก"
en_text = "Diagnosis of acute respiratory tract infections in children"

def get_gemini_embedding(text):
    response = client.models.embed_content(
        model="models/gemini-embedding-2",
        contents=text
    )
    return np.array(response.embeddings[0].values)

def get_bge_embedding(text):
    model = SentenceTransformer("BAAI/bge-m3")
    return model.encode(text, normalize_embeddings=True)

# --- Testing ---
print(f"TH: {th_text}")
print(f"EN: {en_text}\n")

# BGE
bge_th = get_bge_embedding(th_text)
bge_en = get_bge_embedding(en_text)
bge_score = cosine_similarity([bge_th], [bge_en])[0][0]

# Gemini
gem_th = get_gemini_embedding(th_text)
gem_en = get_gemini_embedding(en_text)
gem_score = cosine_similarity([gem_th], [gem_en])[0][0]

# --- Reporting ---
def to_100(score): return round(max(0, score) * 100, 2)

print(f"--- ผลคะแนนความแม่นยำ (0-100) ---")
print(f"BAAI/BGE-M3 (Dense only): {to_100(bge_score)}")
print(f"Gemini Embedding 2      : {to_100(gem_score)}")
print(f"\nวิเคราะห์: {'Gemini 2 ชนะเลิศในด้าน Semantic ข้ามภาษา' if gem_score > bge_score else 'BGE-M3 ยังคงแข็งแกร่งในโหมด Dense'}")

c:\Users\BM MONEY\miniconda3\envs\pharmbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TH: การวินิจฉัยโรคติดเชื้อเฉียบพลันระบบหายใจในเด็ก
EN: Diagnosis of acute respiratory tract infections in children



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 16001.45it/s]


--- ผลคะแนนความแม่นยำ (0-100) ---
BAAI/BGE-M3 (Dense only): 81.9000015258789
Gemini Embedding 2      : 82.49

วิเคราะห์: Gemini 2 ชนะเลิศในด้าน Semantic ข้ามภาษา


In [10]:
import os
import sys
import numpy as np
from google import genai
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from loguru import logger

# ── Setup ────────────────────────────────────────────────────
# ตั้งค่า API Key จาก ENV
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# หัวข้อทดสอบ (Clinical Context ที่มีความกำกวมต่ำและสูง)
TEST_PAIRS = [
    # 1. ศัพท์เทคนิคเฉพาะทาง (Technical Terminology)
    ("ยาต้านจุลชีพกลุ่มเบต้า-แลคแตม", "Beta-lactam antibiotics"),
    
    # 2. อาการที่มีความกำกวม (Symptom Ambiguity) - ทดสอบความเข้าใจบริบท
    ("เด็กมีไข้สูง ซึม ไม่ยอมดื่มนม", "Child with high fever, lethargy, and poor feeding"),
    
    # 3. กระบวนการตัดสินใจทางคลินิก (Clinical Logic)
    ("ข้อบ่งชี้ในการส่งตรวจเอกซเรย์คอมพิวเตอร์", "Indications for performing a CT scan"),
    
    # 4. คำทับศัพท์ที่ภาษาไทยมักใช้ทับศัพท์ภาษาอังกฤษ
    ("ภาวะช็อกจากการติดเชื้อ", "Septic shock"),
    
    # 5. ความกว้างของบริบท (Context Breadth)
    ("แนวทางการพิจารณาปรับยาปฏิชีวนะและการติดตามอาการในผู้ป่วยเด็ก", "Guidelines for antibiotic adjustment and follow-up in pediatric patients")
]

def to_score(score): return round(max(0, score) * 100, 2)

# ── Embedding Engines ────────────────────────────────────────
def get_bge_m3(text):
    model = SentenceTransformer("BAAI/bge-m3")
    return model.encode(text, normalize_embeddings=True)

def get_gemini_2(text, dim=768):
    res = client.models.embed_content(
        model="models/gemini-embedding-2",
        contents=text,
        config={"output_dimensionality": dim}
    )
    return np.array(res.embeddings[0].values)

# ── Runner ──────────────────────────────────────────────────
def run_benchmark():
    logger.info("Starting Embedding Model Benchmark (TH <-> ENG)...")
    
    # Pre-load BGE (เพื่อไม่ให้เสียเวลาโหลดซ้ำในลูป)
    bge_model = SentenceTransformer("BAAI/bge-m3")

    for th, en in TEST_PAIRS:
        print(f"\n{'-'*60}")
        print(f"TH: {th}")
        print(f"EN: {en}")
        print(f"{'-'*60}")

        # 1. BGE-M3 (1024 dim)
        v_bge_th = bge_model.encode(th, normalize_embeddings=True)
        v_bge_en = bge_model.encode(en, normalize_embeddings=True)
        s1 = cosine_similarity([v_bge_th], [v_bge_en])[0][0]

        # 2. Gemini 2 (768 dim)
        v_gem768_th = get_gemini_2(th, 768)
        v_gem768_en = get_gemini_2(en, 768)
        s2 = cosine_similarity([v_gem768_th], [v_gem768_en])[0][0]

        # 3. Gemini 2 (2048 dim)
        v_gem2048_th = get_gemini_2(th, 2048)
        v_gem2048_en = get_gemini_2(en, 2048)
        s3 = cosine_similarity([v_gem2048_th], [v_gem2048_en])[0][0]

        # Log results
        logger.info(f"BGE-M3 (1024)        : {to_score(s1)}/100")
        logger.info(f"Gemini-2 (768)       : {to_score(s2)}/100")
        logger.info(f"Gemini-2 (2048)      : {to_score(s3)}/100")

if __name__ == "__main__":
    run_benchmark()

2026-06-07 00:42:50.834 | INFO     | __main__:run_benchmark:48 - Starting Embedding Model Benchmark (TH <-> ENG)...
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19329.05it/s]



------------------------------------------------------------
TH: ยาต้านจุลชีพกลุ่มเบต้า-แลคแตม
EN: Beta-lactam antibiotics
------------------------------------------------------------


2026-06-07 00:43:10.976 | INFO     | __main__:run_benchmark:75 - BGE-M3 (1024)        : 71.41000366210938/100
2026-06-07 00:43:10.977 | INFO     | __main__:run_benchmark:76 - Gemini-2 (768)       : 84.86/100
2026-06-07 00:43:10.978 | INFO     | __main__:run_benchmark:77 - Gemini-2 (2048)      : 84.67/100



------------------------------------------------------------
TH: เด็กมีไข้สูง ซึม ไม่ยอมดื่มนม
EN: Child with high fever, lethargy, and poor feeding
------------------------------------------------------------


2026-06-07 00:43:12.763 | INFO     | __main__:run_benchmark:75 - BGE-M3 (1024)        : 71.19000244140625/100
2026-06-07 00:43:12.764 | INFO     | __main__:run_benchmark:76 - Gemini-2 (768)       : 78.76/100
2026-06-07 00:43:12.765 | INFO     | __main__:run_benchmark:77 - Gemini-2 (2048)      : 78.17/100


...

2026-06-07 00:43:14.700 | INFO     | __main__:run_benchmark:75 - BGE-M3 (1024)        : 64.9000015258789/100
2026-06-07 00:43:14.701 | INFO     | __main__:run_benchmark:76 - Gemini-2 (768)       : 79.64/100
2026-06-07 00:43:14.702 | INFO     | __main__:run_benchmark:77 - Gemini-2 (2048)      : 79.69/100



------------------------------------------------------------
TH: ภาวะช็อกจากการติดเชื้อ
EN: Septic shock
------------------------------------------------------------


2026-06-07 00:43:16.364 | INFO     | __main__:run_benchmark:75 - BGE-M3 (1024)        : 54.16999816894531/100
2026-06-07 00:43:16.365 | INFO     | __main__:run_benchmark:76 - Gemini-2 (768)       : 84.77/100
2026-06-07 00:43:16.365 | INFO     | __main__:run_benchmark:77 - Gemini-2 (2048)      : 84.72/100



------------------------------------------------------------
TH: แนวทางการพิจารณาปรับยาปฏิชีวนะและการติดตามอาการในผู้ป่วยเด็ก
EN: Guidelines for antibiotic adjustment and follow-up in pediatric patients
------------------------------------------------------------


2026-06-07 00:43:18.213 | INFO     | __main__:run_benchmark:75 - BGE-M3 (1024)        : 78.73999786376953/100
2026-06-07 00:43:18.213 | INFO     | __main__:run_benchmark:76 - Gemini-2 (768)       : 85.61/100
2026-06-07 00:43:18.214 | INFO     | __main__:run_benchmark:77 - Gemini-2 (2048)      : 85.34/100


In [1]:
import os
print(os.getenv("INGEST_VISION_MAX_TOKENS"))

8192


In [2]:
import subprocess

result = subprocess.run(
    [
        "python",
        "-m",
        "scripts.debug_backend",
        "--step",
        "retrieve",
        "--query",
        "ไอมา 3 วัน มีน้ำมูกใส ไม่มีไข้"
    ],
    capture_output=True,
    text=True,
    encoding="utf-8"
)

print(result.stdout)


══════════════════════════════════════════════════
  PharmBot Backend Debug
  Query: ไอมา 3 วัน มีน้ำมูกใส ไม่มีไข้
══════════════════════════════════════════════════

──────────────────────────────────────────────────
  Step 3: Qdrant Retrieval
──────────────────────────────────────────────────
✓ Retrieved 4 chunks
→ [1] score=0.657  source=Thai URI Children, p.14, [vision]
→      text=## แผนภูมิที่ 1: แนวทางการประเมินผู้ป่วยเด็กที่มีน้ำมูก ไอ ± ไข้

**จุดเริ่มต้น**: เด็กมีน้ำมูก ไอ ±...
→ [2] score=0.6403  source=Thai URI Children, p.16, [vision]
→      text=อาการของโรคหวัด ขึ้นกับอายุและชนิดของเชื้อไวรัส เด็กเล็กอาจมีไข้และน้ำมูกเป็นอาการเด่น เด็กโตมักไม่ม...
→ [3] score=0.6325  source=Thai URI Children, p.17, [vision]
→      text=16

ผู้ป่วยเด็ก โรคหวัดอาจเป็นอาการเริ่มต้นของโรคติดเชื้อทางเดินหายใจทั้งหมด ที่เกิดจากเชื้อไวรัส หา...
→ [4] score=0.6301  source=AAFP_2022_Original (2022), [docling_text]
→      text=## Common Cold

Although the common cold is typically a mild, self-lim

In [3]:
import subprocess

result = subprocess.run(
    [
        "python",
        "-m",
        "scripts.debug_backend",
        "--step",
        "retrieve",
        "--query",
        "แพ้ยา amoxicillin จะใช้ยาอะไรแทน"
    ],
    capture_output=True,
    text=True,
    encoding="utf-8"
)

print(result.stdout)


══════════════════════════════════════════════════
  PharmBot Backend Debug
  Query: แพ้ยา amoxicillin จะใช้ยาอะไรแทน
══════════════════════════════════════════════════

──────────────────────────────────────────────────
  Step 3: Qdrant Retrieval
──────────────────────────────────────────────────
✓ Retrieved 4 chunks
→ [1] score=0.6435  source=Thai URI Children, p.54, [vision]
→      text=## แนวทางการพิจารณาปรับยาปฏิชีวนะและการติดตามอาการ

แนวทางการดูแลรักษาโรคติดเชื้อเฉียบพลันระบบหายใจใ...
→ [2] score=0.6255  source=Thai URI Children, p.56, [vision]
→      text=## ตารางที่ 4: ขนาดและระยะเวลาการให้ยาต้านจุลชีพแบบรับประทานเบื้องต้นสำหรับ AOM

| ยา | ขนาดยา | ระย...
→ [3] score=0.6241  source=Thai URI Children, p.56, [vision]
→      text=แนวทางการดูแลรักษาโรคติดเชื้อเฉียบพลันระบบหายใจในเด็ก พ.ศ. 2562 55

* กรณีสงสัยเชื้อ S. pneumoniae ด...
→ [4] score=0.6181  source=Thai URI Children, p.40, [vision]
→      text=## ตารางที่ 3: ขนาดและระยะเวลาการให้ยาต้านจุลชีพแบบรับประทานเบื้องต้นสำหรับ

In [4]:
import subprocess

result = subprocess.run(
    [
        "python",
        "-m",
        "scripts.debug_backend",
        "--step",
        "retrieve",
        "--query",
        "paracetamol ขนาดยาสำหรับผู้ใหญ่"
    ],
    capture_output=True,
    text=True,
    encoding="utf-8"
)

print(result.stdout)


══════════════════════════════════════════════════
  PharmBot Backend Debug
  Query: paracetamol ขนาดยาสำหรับผู้ใหญ่
══════════════════════════════════════════════════

──────────────────────────────────────────────────
  Step 3: Qdrant Retrieval
──────────────────────────────────────────────────
✓ Retrieved 4 chunks
→ [1] score=0.6187  source=Thai URI Children, p.24, [vision]
→      text=## ตารางที่ 2: ขนาดและระยะเวลาการให้ยาต้านจุลชีพสำหรับ Group A streptococcal pharyngitis⁴,¹¹-¹⁴
| ยา...
→ [2] score=0.5894  source=Thai URI Children, p.54, [vision]
→      text=## แนวทางการพิจารณาปรับยาปฏิชีวนะและการติดตามอาการ

แนวทางการดูแลรักษาโรคติดเชื้อเฉียบพลันระบบหายใจใ...
→ [3] score=0.5738  source=AAFP_2022_Original (2022), [docling_text]
→      text=## Appropriate Antibiotic Dosing for Outpatient Treatment of Upper Respiratory Tract Infections [......
→ [4] score=0.5622  source=Thai URI Children, p.40, [vision]
→      text=## ตารางที่ 3: ขนาดและระยะเวลาการให้ยาต้านจุลชีพแบบรับประทานเบื้องต้

# Test Case Data

In [5]:
import pandas as pd


positive = pd.read_csv("data/test-cases/positive_cases.csv", encoding="utf-8")
negative = pd.read_csv("data/test-cases/negative_cases.csv", encoding="utf-8")
incomplete = pd.read_csv("data/test-cases/incomplete_cases.csv", encoding="utf-8")

In [13]:
pd.set_option("display.max_colwidth", None)

In [14]:
positive.head()

,id,category,input,expected_output,reference
0,easy_1,easy,แม่พาลูกชายอายุ 3 ขวบมาร้านยา บอกว่าเป็นหวัดมา 2 วัน มีน้ำมูกใส ไอเล็กน้อย ไข้ 37.8°C ไม่มีหูอื้อ ไม่เจ็บคอมาก ไม่มีน้ำมูกข้นเขียว แม่ขอ 'ยาแก้อักเสบ' เพราะคิดว่าจะทำให้หายเร็วขึ้น,"ยาที่ให้ได้: Paracetamol สำหรับลดไข้, น้ำเกลือหยอดจมูก, ดูดน้ำมูกด้วยลูกยาง ห้ามให้เด็ก <4 ปี: ยาแก้ไอ-ยาลดน้ำมูก-ยาแก้คัดจมูก (antihistamine, pseudoephedrine, dextromethorphan) เพราะไม่มีประโยชน์และมีผลข้างเคียง กระสับกระส่าย ประสาทหลอน ห้ามให้ยาปฏิชีวนะ (ยาแก้อักเสบ) เพราะไม่มีข้อบ่งชี้","AAFP 2022 P.628, table 1"
1,easy_2,easy,ชายอายุ 25 ปีมาร้านยาบอกว่าเสียงแหบมา 3 วัน คัดจมูกเล็กน้อย ไม่มีไข้ ไม่มีหายใจลำบาก เจ็บคอเล็กน้อย ไม่มีน้ำลายไหล เขาพูดว่าจะต้องนำเสนองานในสัปดาห์หน้าและอยากได้ยาปฏิชีวนะเพื่อให้หายเร็วขึ้น,"Supportive care: พักเสียง ดื่มน้ำมาก ใช้ไอน้ำ ไม่ควรกระซิบ (ทำให้สายเสียงตึงกว่า), ยาแก้ปวด paracetamol ถ้ามีอาการ โรคมักหายเองใน 1-2 สัปดาห์ ห้ามให้ยาปฏิชีวนะ เพราะโรคนี้มักเกิดจากไวรัส ประโยชน์ไม่คุ้มค่า","AAFP 2022 P.628, table 1"
2,easy_3,easy,หญิงอายุ 30 ปี เจ็บคอมา 1 วัน มีน้ำมูกใส ไอมาก มีน้ำตาไหล ไข้ 37.5°C ไม่มีต่อมน้ำเหลืองที่คอโต ไม่มีหนองที่ทอนซิล เธอขอซื้อยา amoxicillin เพราะเคยใช้แล้วหายดี,"Supportive care: paracetamol และน้ำอุ่นดื่มมากๆ ห้ามให้ amoxicillin เพราะอาการบ่งชี้ viral pharyngitis (ไอมาก, น้ำมูกใส, น้ำตาไหล) ซึ่งยาปฏิชีวนะไม่ได้ผล","AAFP 2022 P.628, table 1"
3,easy_4,easy,แม่พาลูกสาว 5 ขวบมาร้านยา น้ำมูกเปลี่ยนเป็นสีเขียวมา 1 วัน (เริ่มป่วย 3 วันก่อน) ไข้ 38°C ไม่ปวดหน้าผาก ไม่ปวดแก้ม ไม่เจ็บคอ ไม่ปวดหู แม่บอก 'น้ำมูกเขียวแสดงว่ามีเชื้อแบคทีเรียแน่นอน ต้องให้ยาฆ่าเชื้อ',"Supportive care: Paracetamol ลดไข้, น้ำเกลือหยอดจมูก, ดื่มน้ำมาก, สังเกตอาการ นัดกลับมาหรือไปพบแพทย์ถ้าอาการไม่ดีขึ้นใน 10 วัน หรือแย่ลงกะทันหัน ห้ามให้ยาปฏิชีวนะ เพราะน้ำมูกเขียวเป็นการดำเนินโรคปกติของไวรัส ไม่ใช่เกณฑ์ติดเชื้อแบคทีเรีย","แนวทางดูแลรักษาโรคติดเชื้อเฉียบพลันระบบหายใจในเด็ก พ.ศ. 2562 P.29, 33"
4,easy_5,easy,ผู้ป่วยผู้ใหญ่มีอาการไข้ต่ำๆ ปวดกล้ามเนื้อ ปวดศีรษะ คัดจมูก มีน้ำมูกใส และไอมาเป็นเวลา 3 วัน ไม่มีอาการรุนแรงอื่นใด,"ให้การรักษาตามอาการ (Supportive care): ยาแก้ปวดลดไข้ paracetamol 500 mg prn q4-6h ไม่เกินวันละ 4000 mg และยาสูตรผสมต้านฮิสตามีน/ลดน้ำมูก (Chlorpheniramine maleate 4 mg, Phenylephrine HCl 10 mg เช่น Nasolin รับประทานครั้งละ 1 เม็ด วันละ 3 ครั้ง / Sulidine CP ครั้งละ 1 เม็ด ทุก 4-6 ชั่วโมง) ห้ามสั่งจ่ายยาปฏิชีวนะ",AAFP 2022 table 1
